In [1]:
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build

SCOPES = ['https://www.googleapis.com/auth/calendar',
          'https://www.googleapis.com/auth/gmail.modify',
          'https://www.googleapis.com/auth/presentations',
          'https://www.googleapis.com/auth/gmail.readonly']

In [2]:
def generate_rrule(weekdays, repeat_until=None):
    day_map = {
        "Monday": "MO",
        "Tuesday": "TU",
        "Wednesday": "WE",
        "Thursday": "TH",
        "Friday": "FR",
        "Saturday": "SA",
        "Sunday": "SU"
    }
    byday = [day_map[day] for day in weekdays if day in day_map]

    if len(byday) == 7 or len(byday) == 0:
        rule = "RRULE:FREQ=DAILY"
    else:
        rule = f"RRULE:FREQ=WEEKLY;BYDAY={','.join(byday)}"

    if repeat_until:
        rule += f";UNTIL={repeat_until.strftime('%Y%m%dT%H%M%SZ')}"

    return rule


In [3]:

from datetime import datetime

# Define your scope

def schedule_daily_habit(summary, start_time, end_time, attendees_emails=[], repeat_until=None, days =[]):# Schedules daily habits for user into calendar
    # Authenticate
    flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
    creds = flow.run_local_server(port=8080)
    service = build('calendar', 'v3', credentials=creds)

    # Recurrence rule: DAILY
    
    recurrence_rule = generate_rrule(days, repeat_until)
    if repeat_until:
        recurrence_rule += f";UNTIL={repeat_until.strftime('%Y%m%dT%H%M%SZ')}"

    event = {
        'summary': summary,
        'reminders': {
            'useDefault': False,
            'overrides': [
                {'method': 'popup', 'minutes':0 },
                
            ]
        },
        'start': {'dateTime': start_time.isoformat(), 'timeZone': 'UTC'},
        'end': {'dateTime': end_time.isoformat(), 'timeZone': 'UTC'},
        'attendees': [{'email': email} for email in attendees_emails],
        'recurrence': [recurrence_rule],
    }

    created_event = service.events().insert(
        calendarId='primary',
        body=event
    ).execute()

    return created_event.get('htmlLink', 'Recurring event created, but no link returned.')

In [4]:
def create_google_meet_event(summary, start_time, end_time): # Schedules tasks for the user
    # Authenticate
    flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)

    creds = flow.run_local_server(port=8080)
    service = build('calendar', 'v3', credentials=creds)
    
    event = {
        'summary': summary,
        'reminders': {
            'useDefault': False,
            'overrides': [
                {'method': 'popup', 'minutes': 10},
                {'method': 'email', 'minutes': 30}
            ]
        },
        'start': {'dateTime': start_time.isoformat(), 'timeZone': 'UTC'},
        'end': {'dateTime': end_time.isoformat(), 'timeZone': 'UTC'},
    }

    created_event = service.events().insert(
        calendarId='primary',
        body=event,
        conferenceDataVersion=1
    ).execute()

    return created_event.get('htmlLink', 'No event created.')

In [9]:
def reschedule_event(event_name, new_start_time, new_end_time):
    # Authenticate
    flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
    creds = flow.run_local_server(port=8080)
    service = build('calendar', 'v3', credentials=creds)
    
    # Search for the event by name (no ordering if singleEvents is False)
    events_result = service.events().list(
        calendarId='primary',
        q=event_name,
        singleEvents=False,  # This includes recurring master events
        maxResults=10
    ).execute()
    
    events = events_result.get('items', [])
    
    if not events:
        return f"No events found with name: {event_name}"
    
    # Get the first matching event
    event = events[0]
    event_id = event['id']
    
    # Update the start and end time
    event['start']['dateTime'] = new_start_time.isoformat()
    event['end']['dateTime'] = new_end_time.isoformat()
    event['start']['timeZone'] = 'UTC'
    event['end']['timeZone'] = 'UTC'

    # Update the event
    updated_event = service.events().update(
        calendarId='primary',
        eventId=event_id,
        body=event
    ).execute()
    
    return updated_event.get('htmlLink', 'Event rescheduled, but no link returned.')

In [10]:
reschedule_event("Learning reminder", datetime(2025, 10, 20, 10, 0), datetime(2025, 10, 20, 11, 0))

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=893172960263-vp9ogeuo8fetkmv49m9su7c8s8b7b430.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8080%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcalendar+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.modify+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fpresentations+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.readonly&state=pTR1HUHSEhfsovauwLINt0KrWQiH1I&access_type=offline


'https://www.google.com/calendar/event?eid=aGRxb2VzbnFhYmY2MTBsb2pmcHA3MXZ1NWMgc29kaGkua3Jpc2gwNUBt'

In [11]:
from datetime import timedelta
start = datetime.utcnow().replace(hour=6, minute=0, second=0, microsecond=0)
end = start + timedelta(minutes=30)
repeat_until = start + timedelta(days=30)
schedule_daily_habit("Work out",start_time=start,end_time=end)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_35740\1128858441.py:2: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  start = datetime.utcnow().replace(hour=6, minute=0, second=0, microsecond=0)


Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=893172960263-vp9ogeuo8fetkmv49m9su7c8s8b7b430.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8080%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcalendar+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.modify+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fpresentations+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.readonly&state=2JGtjfqNWuwQ3DTHlYlYWLu8FPStTL&access_type=offline


'https://www.google.com/calendar/event?eid=bGV0ZGJtbGV1NnNtNWxldTJsZDVhZm44bGtfMjAyNTA0MTlUMDYwMDAwWiBzb2RoaS5rcmlzaDA1QG0'

In [12]:
from datetime import timedelta
start = datetime.utcnow().replace(hour=6, minute=0, second=0, microsecond=0)
end = start + timedelta(minutes=30)
repeat_until = start + timedelta(days=30)
create_google_meet_event("study",start_time=start,end_time=end)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_35740\3981108211.py:2: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  start = datetime.utcnow().replace(hour=6, minute=0, second=0, microsecond=0)


Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=893172960263-vp9ogeuo8fetkmv49m9su7c8s8b7b430.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8080%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcalendar+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.modify+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fpresentations+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.readonly&state=kmto83q0u7eSuGfEnGKF5QttAgDITk&access_type=offline


'https://www.google.com/calendar/event?eid=NWcyYzkwMWtpYzdzNDExamtkM3RxbzRic2cgc29kaGkua3Jpc2gwNUBt'

In [13]:
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from datetime import datetime, timedelta

SCOPES = ['https://www.googleapis.com/auth/calendar.readonly']

def list_events_for_day():
    target_date = datetime.utcnow().replace(hour=0, minute=0, second=0, microsecond=0) #today
    # Authenticate
    flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
    creds = flow.run_local_server(port=8080)
    service = build('calendar', 'v3', credentials=creds)

    # Define start and end of the day in UTC
    start_of_day = datetime.combine(target_date, datetime.min.time()).isoformat() + 'Z'
    end_of_day = datetime.combine(target_date, datetime.max.time()).isoformat() + 'Z'

    # Get events
    events_result = service.events().list(
        calendarId='primary',
        timeMin=start_of_day,
        timeMax=end_of_day,
        singleEvents=True,
        orderBy='startTime'
    ).execute()

    events = events_result.get('items', [])

    clean_events = []
    for event in events:
        event_type = 'habit' if 'recurringEventId' in event or 'recurrence' in event else 'task'
        clean_events.append({
            'summary': event.get('summary', 'No Title'),
            'reminders': {
                'useDefault': False,
                'overrides': [
                    {'method': 'popup', 'minutes': 10},
                    {'method': 'email', 'minutes': 30}
                ]
            },
            'start': event['start'].get('dateTime', event['start'].get('date')),
            'end': event['end'].get('dateTime', event['end'].get('date')),
            'event_type': event_type
        })

    return clean_events

In [14]:
calendar_events = list_events_for_day()

C:\Users\ASUS\AppData\Local\Temp\ipykernel_35740\743448217.py:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  target_date = datetime.utcnow().replace(hour=0, minute=0, second=0, microsecond=0) #today


Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=893172960263-vp9ogeuo8fetkmv49m9su7c8s8b7b430.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8080%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcalendar.readonly&state=7FHjOss9tOARR9jpJXD2aIO9lJVOCF&access_type=offline


In [15]:
calendar_events

[{'summary': 'study',
  'reminders': {'useDefault': False,
   'overrides': [{'method': 'popup', 'minutes': 10},
    {'method': 'email', 'minutes': 30}]},
  'start': '2025-04-19T11:30:00+05:30',
  'end': '2025-04-19T12:00:00+05:30',
  'event_type': 'task'},
 {'summary': 'Work out',
  'reminders': {'useDefault': False,
   'overrides': [{'method': 'popup', 'minutes': 10},
    {'method': 'email', 'minutes': 30}]},
  'start': '2025-04-19T11:30:00+05:30',
  'end': '2025-04-19T12:00:00+05:30',
  'event_type': 'habit'},
 {'summary': 'study',
  'reminders': {'useDefault': False,
   'overrides': [{'method': 'popup', 'minutes': 10},
    {'method': 'email', 'minutes': 30}]},
  'start': '2025-04-19T11:30:00+05:30',
  'end': '2025-04-19T12:00:00+05:30',
  'event_type': 'task'},
 {'summary': 'Work out',
  'reminders': {'useDefault': False,
   'overrides': [{'method': 'popup', 'minutes': 10},
    {'method': 'email', 'minutes': 30}]},
  'start': '2025-04-19T11:30:00+05:30',
  'end': '2025-04-19T12:00:0

In [ ]:
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from datetime import datetime, timedelta
import pytz
from plyer import notification  # pip install plyer

SCOPES = ['https://www.googleapis.com/auth/calendar.readonly']

def notify_today_events():
    # Authenticate
    flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
    creds = flow.run_local_server(port=8080)
    service = build('calendar', 'v3', credentials=creds)

    # Define start and end of the current day in UTC
    utc = pytz.UTC
    now = datetime.utcnow().replace(tzinfo=utc)
    start_of_day = datetime(now.year, now.month, now.day, 0, 0, 0, tzinfo=utc)
    end_of_day = start_of_day + timedelta(days=1)

    # Fetch today's events
    events_result = service.events().list(
        calendarId='primary',
        timeMin=start_of_day.isoformat(),
        timeMax=end_of_day.isoformat(),
        singleEvents=True,
        orderBy='startTime'
    ).execute()
    
    events = events_result.get('items', [])

    if not events:
        notification.notify(
            title='Google Calendar',
            message='No events scheduled for today.',
            timeout=10
        )
        return

    # Notify for each event
    for event in events:
        start = event['start'].get('dateTime', event['start'].get('date'))
        summary = event.get('summary', 'No Title')
        notification.notify(
            title='Upcoming Event Today',
            message=f"{summary} at {start}",
            timeout=10  # seconds
        )

if __name__ == "__main__":
    notify_today_events()